# JEPA (Joint Embedding Predictive Architecture) Tutorial

This notebook demonstrates how to train and use JEPA for self-supervised visual representation learning.

## What is JEPA?

JEPA is a self-supervised learning method that predicts parts of a latent representation from other parts. Key components:

- **Target Encoder**: Encodes the full image (frozen)
- **Context Encoder**: Encodes partially masked views
- **Predictor**: Predicts masked regions from the context
- **Multi-Block Masking**: Uses spatial and temporal masking for robust learning

JEPA excels at learning rich visual representations without requiring labeled data.

Note: JEPA typically requires distributed training (multiple GPUs). This notebook demonstrates the setup and a simplified training approach.

In [ ]:
# Setup and Imports
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Configuration

JEPA uses the `JEPAConfig` class for configuration.

In [ ]:
from world_models.configs.jepa_config import JEPAConfig

# Create JEPA configuration
config = JEPAConfig()

# Adjust for simpler demo (smaller model, fewer epochs)
config.epochs = 2
config.batch_size = 8
config.crop_size = 128  # Smaller for demo
config.patch_size = 16
config.pred_depth = 4
config.pred_emb_dim = 256

# Use CIFAR-10 for faster training
config.dataset = "cifar10"
config.data_root = "./data"

# Disable wandb for demo
config.enable_wandb = False

print("=== JEPA Configuration ===")
print(f"Model: {config.model_name}")
print(f"Crop size: {config.crop_size}")
print(f"Patch size: {config.patch_size}")
print(f"Embedding dim: {config.pred_emb_dim}")
print(f"Predictor depth: {config.pred_depth}")
print(f"Epochs: {config.epochs}")
print(f"Batch size: {config.batch_size}")
print(f"Dataset: {config.dataset}")

## Understanding JEPA Components

JEPA consists of three main components: encoder, predictor, and target encoder.

In [ ]:
# Import JEPA models
from world_models.models.jepa_agent import JEPAAgent
from world_models.inference.operators import JEPAOperator

print("=== JEPA Architecture ===")
print("JEPA uses Vision Transformer (ViT) based components:")
print("- Target Encoder: ViT encoder (frozen, encodes full images)")
print("- Context Encoder: ViT encoder (trainable, encodes masked views)")
print("- Predictor: ViT decoder that predicts latent features for masked patches")

# Model parameters from config
num_patches = (config.crop_size // config.patch_size) ** 2
print(f"\nNumber of patches: {num_patches}")
print(f"Mask ratio: {config.pred_mask_scale}")
print(f"Context blocks: {config.num_enc_masks}")
print(f"Target blocks: {config.num_pred_masks}")

## JEPA Training

JEPA training involves:

1. **Data Preparation**: Apply augmentations (color jitter, gaussian blur, etc.)
2. **Multi-Block Masking**: Generate context and target blocks
3. **Forward Pass**: 
   - Target encoder processes full image → target features
   - Context encoder processes masked image → context features
   - Predictor predicts target features from context
4. **Loss**: Smooth L1 loss between predicted and target features
5. **Momentum Update**: Target encoder updated via exponential moving average of context encoder

Note: JEPA requires distributed training for large-scale training. We'll demonstrate a simplified single-GPU version.

In [ ]:
# Create JEPA agent for demonstration
# In practice, JEPA requires distributed training setup
print("Creating JEPA agent...")

try:
    agent = JEPAAgent(config)
    print("JEPA agent created successfully!")
    
    # Show model components
    print("\n=== JEPA Components ===")
    print(f"Encoder: {type(agent.encoder).__name__}")
    print(f"Predictor: {type(agent.predictor).__name__}")
    print(f"Target Encoder: {type(agent.target_encoder).__name__}")
    
except Exception as e:
    print(f"Note: Full JEPA training requires distributed setup")
    print(f"Error: {e}")
    print("\nWe'll demonstrate JEPA preprocessing instead.")

In [ ]:
# Demonstrate JEPA preprocessing and masking
print("=== JEPA Preprocessing Demo ===")

from world_models.masks.multiblock import MaskCollator

# Create mask collator
mask_collator = MaskCollator(
    input_size=config.crop_size,
    patch_size=config.patch_size,
    pred_mask_scale=config.pred_mask_scale,
    enc_mask_scale=config.enc_mask_scale,
    aspect_ratio=(1.0,),
    nenc=config.num_enc_masks,
    npred=config.num_pred_masks,
    allow_overlap=False,
    min_keep=config.min_keep,
)

# Create dummy images
batch_size = 4
dummy_images = torch.randn(batch_size, 3, config.crop_size, config.crop_size)

# Generate masks
imgs, masks_enc, masks_pred = mask_collator(dummy_images)

print(f"Batch size: {batch_size}")
print(f"Image shape: {dummy_images.shape}")
print(f"Number of context masks: {len(masks_enc)}")
print(f"Number of target masks: {len(masks_pred)}")
print(f"Context mask shape: {masks_enc[0].shape}")
print(f"Target mask shape: {masks_pred[0].shape}")

In [ ]:
# Visualize masking
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Show original image
axes[0].imshow(dummy_images[0].permute(1, 2, 0).numpy())
axes[0].set_title('Original Image')
axes[0].axis('off')

# Create context mask visualization
patch_grid_size = config.crop_size // config.patch_size
context_mask_grid = masks_enc[0][0].reshape(patch_grid_size, patch_grid_size).float()
target_mask_grid = masks_pred[0][0].reshape(patch_grid_size, patch_grid_size).float()

axes[1].imshow(context_mask_grid.numpy(), cmap='Blues', alpha=0.7)
axes[1].set_title('Context Mask (visible)')
axes[1].axis('off')

axes[2].imshow(target_mask_grid.numpy(), cmap='Reds', alpha=0.7)
axes[2].set_title('Target Mask (predicted)')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('jepa_masks.png', dpi=150)
plt.show()

print("Mask visualization saved to jepa_masks.png")

## JEPA Inference Operator

The `JEPAOperator` handles preprocessing for inference, including image normalization and masking.

In [ ]:
# Using JEPAOperator for preprocessing
from PIL import Image

# Create operator
jepa_operator = JEPAOperator(
    image_size=224,
    patch_size=16,
    mask_ratio=0.75
)

# Example preprocessing
dummy_image = Image.new('RGB', (224, 224), color='green')
processed = jepa_operator.process({'images': [dummy_image]})

print("=== JEPA Operator Demo ===")
print("Processed images shape:", processed['images'].shape)
print("Mask shape:", processed['mask'].shape)
print("Image value range:", processed['images'].min().item(), "to", processed['images'].max().item())

## JEPA for Representation Learning

JEPA learns representations by predicting latent features, which captures semantic information.

In [ ]:
# Show training command for full JEPA training
print("=== Full JEPA Training (Distributed) ===")
print("""
For full JEPA training on ImageNet or large datasets:

1. Using torchrun (recommended):
```bash
torchrun --nproc_per_node=8 world_models/training/train_jepa.py \
    --config_path configs/jepa_imagenet.yaml
```

2. Using SLURM:
```bash
srun --ntasks=8 --gres=gpu:8 python world_models/training/train_jepa.py
```

3. Configuration options:
   - model_name: vit_small, vit_base, vit_large
   - crop_size: 224 (default), 168 (smaller)
   - pred_depth: 6-12 (predictor layers)
   - pred_emb_dim: 384-768 (predictor embedding dim)
   - epochs: 100-800
   - batch_size: 64-256 per GPU
""")

## Advanced Configuration

JEPA has many configurable parameters for experimentation.

In [ ]:
# Explore JEPA config options
print("=== JEPA Advanced Configuration ===")

# Masking options
print("\nMasking:")
print(f"  Patch size: {config.patch_size}")
print(f"  Context mask scale: {config.enc_mask_scale}")
print(f"  Target mask scale: {config.pred_mask_scale}")
print(f"  Min keep patches: {config.min_keep}")
print(f"  Allow overlap: {config.allow_overlap}")

# Augmentations
print("\nAugmentations:")
print(f"  Gaussian blur: {config.use_gaussian_blur}")
print(f"  Horizontal flip: {config.use_horizontal_flip}")
print(f"  Color distortion: {config.use_color_distortion}")
print(f"  Color jitter strength: {config.color_jitter_strength}")

# Optimization
print("\nOptimization:")
print(f"  Learning rate: {config.lr}")
print(f"  Weight decay: {config.weight_decay}")
print(f"  Warmup epochs: {config.warmup}")
print(f"  EMA momentum: {config.ema}")

# Logging
print("\nLogging:")
print(f"  Enable WandB: {config.enable_wandb}")
print(f"  Project: {config.wandb_project}")

## Summary

In this tutorial, you learned:

1. **JEPA Basics**: Understanding the Joint Embedding Predictive Architecture
2. **Architecture**: Target encoder, context encoder, and predictor components
3. **Training**: Multi-block masking and self-supervised learning objectives
4. **Inference**: Using JEPAOperator for preprocessing
5. **Configuration**: Various hyperparameters for customization

### Key Takeaways

- JEPA learns by predicting latent features rather than pixel values
- Uses multi-block masking for robust representation learning
- Requires distributed training for large-scale datasets
- Learned features are suitable for various downstream vision tasks

### Next Steps

- Run full distributed training on ImageNet
- Experiment with different masking strategies
- Try different encoder architectures (ViT-S, ViT-B, ViT-L)
- Fine-tune on downstream tasks